# Semana 05: Testes Unitários, de Integração e Quality Gates no CI

## Módulo de Automação de Qualidade — Fábrica Virtual Smart N1

Este notebook apresenta a implementação de **Testes Unitários e de Integração** automatizados no pipeline de CI com **Pytest**, a medição de **cobertura de código (Code Coverage)** e a criação de **Quality Gates** para garantir a estabilidade do software fabril.

### Objetivos de aprendizagem
- Compreender a Pirâmide de Testes (Unitários, Integração e E2E).
- Escrever suítes de testes unitários e testes de integração utilizando fixtures do `pytest`.
- Simular dependências externas (como bancos de dados e conectores MQTT) utilizando *Mocks*.
- Mensurar e gerar relatórios de cobertura de código com `coverage.py` / `pytest-cov`.
- Configurar **Quality Gates** para barrar Pull Requests que não atinjam a meta de testes.
- Executar um avaliador de suíte de testes em Python.

---


## 1. Fundamentação Teórica

### 1.1 Testes Unitários vs Testes de Integração

```text
  +-----------------------------------+     +-----------------------------------+
  |         TESTES UNITÁRIOS          |     |       TESTES DE INTEGRAÇÃO        |
  |                                   |     |                                   |
  | Isolam uma função/método.         |     | Validam a interação real entre    |
  | Usam Mocks para eliminar I/O,     |     | o módulo de código, banco de      |
  | banco de dados e conexões.        |     | dados e serviços de terceiros.    |
  | Executam em milissegundos.        |     | Executam em segundos.             |
  +-----------------------------------+     +-----------------------------------+
```

---

### 1.2 O Conceito de Quality Gate no CI

O **Quality Gate** funciona como uma trava automatizada no pipeline do GitHub Actions:

```text
  [Pull Request aberto para develop]
                 |
                 v
  [Job de CI executa Pytest & Coverage]
                 |
                 v
     +-----------------------+
     |  CRITÉRIO DO GATE:    |
     |  1. Failures = 0      |
     |  2. Coverage >= 80%   |
     +-----------------------+
         /               \
     (PASS)             (FAIL)
       /                   \
      v                     v
 [Merge Habilitado]   [Build Bloqueado no GitHub]
```

---


## 2. Prática — Suíte de Testes Unitários, Integração e Quality Gate em Python

Nesta atividade prática, implementaremos uma função de cálculo de eficiência de linha de produção, sua suíte de testes (com mock de banco de dados) e o validador de Quality Gate.

In [ ]:
class BancoDadosFabrilMock:
    def buscar_pecas_produzidas(self, linha_id):
        # Simulação de consulta SQL ao banco de dados fabril
        return {"linha_id": linha_id, "pecas_ok": 950, "pecas_refugadas": 50, "meta_producao": 1000}

def calcular_eficiencia_linha(banco_mock, linha_id):
    dados = banco_mock.buscar_pecas_produzidas(linha_id)
    total = dados["pecas_ok"] + dados["pecas_refugadas"]
    if total == 0:
        return 0.0
    eficiencia_pct = (dados["pecas_ok"] / dados["meta_producao"]) * 100.0
    return round(eficiencia_pct, 1)

# Execução da suíte de testes unitários e de integração
def rodar_pipeline_testes():
    db_mock = BancoDadosFabrilMock()
    
    # Teste 1: Teste de integração com Mock DB
    res_eficiencia = calcular_eficiencia_linha(db_mock, "LINHA_01")
    teste1_ok = res_eficiencia == 95.0
    
    testes_totais = 1
    testes_passados = 1 if teste1_ok else 0
    cobertura_pct = 92.5
    meta_gate_pct = 80.0
    
    gate_aprovado = (testes_passados == testes_totais) and (cobertura_pct >= meta_gate_pct)
    
    return {
        "testes_passados": f"{testes_passados}/{testes_totais}",
        "eficiencia_calculada": f"{res_eficiencia}%",
        "cobertura_codigo": f"{cobertura_pct}%",
        "meta_quality_gate": f"{meta_gate_pct}%",
        "status_pipeline": "SUCESSO (MERGE PERMITIDO)" if gate_aprovado else "FALHA (BUILD BLOQUEADO)"
    }

relatorio_testes = rodar_pipeline_testes()
print("=== RELATÓRIO DO PIPELINE DE TESTES E QUALITY GATE ===\n")
for k, v in relatorio_testes.items():
    print(f"- {k}: {v}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Explique a função do uso de **Mocks** (ou *Stubs*) em testes unitários. Por que não devemos conectar a testes unitários um banco de dados PostgreSQL em produção ou uma API externa real?

### Questão 2
Qual a diferença fundamental entre um **Teste Unitário** e um **Teste de Integração**? Dê um exemplo de teste de integração no contexto da fábrica Smart N1.

### Questão 3
Como configurar a instrução no arquivo de workflow `.github/workflows/ci.yml` para que a execução do Pytest falhe o build se a cobertura total de código for menor que 85%?
